In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Symbolic Differentiation

In this notebook our goal is to implement *symbolic differentiation*.  Concretely, we want to implement a function `diff` that takes one argument:
  - The argument `e` represents an *arithmetic expression*.
    Here an arithmetic expression is built from variables and numbers by application
    of any of the operator symbols "`+`", "`-`", "`*`", "`/`", and "`**`".
    The operator "`**`" represents exponentiation, i.e. an expression of the form 
    $a \texttt{**} b$ is interpreted as $a^b$.          
    Furthermore, if $e$ is an expression, then both $\exp(e)$ and $\ln(e)$ are expressions too.

The function call `diff e` will then compute the derivative of `e` with respect to the variable `x`.  For example, for the expression `x * exp(x)` the function will compute the output
`1*exp(x) + x*(1*exp(x))` because we have:
$$ \frac{\mathrm{d}\;}{\mathrm{d}x} \bigl( x \cdot \mathrm{e}^x \bigr) = 1 \cdot \mathrm{e}^x + x \cdot \mathrm{e}^x. $$

It would be very tedious to **represent** arithmetic expressions as strings.  Instead, we will represent arithmetic expressions as values of the discriminated union `Expr` that is defined in the notebook `Expr-Parser.ipynb`:
```
type Expr =
    | Num of int             // number
    | Var of string          // variable
    | Add of Expr * Expr     // +
    | Sub of Expr * Expr     // -
    | Mul of Expr * Expr     // *
    | Div of Expr * Expr     // /
    | Pow of Expr * Expr     // **
    | App of string * Expr   // application of a function like exp or ln
```
For example, the arithmetic expression `x*exp(x)` is represented as  
```
Mul (Var "x", App ("exp", Var "x"))
```
In order to be able to convert strings into expressions, we need a *parser*.  Therefore we import the notebook `Expr-Parser.ipynb`.

In [2]:
importNotebook "Expr-Parser.ipynb"

imported 

Expr-Parser.ipynb

This notebook provides the function `parse`.  Given a string $s$, the call `parse s` returns the expression represented by $s$.  The following example clarifies this:

In [3]:
parse "x*exp(x)"

Mul (Var "x", App ("exp", Var "x"))

Now we are ready to implement the function `diff`.  It takes one argument:
- `e` is an arithmetic expression.

It returns the derivative of `e` with respect to the variable `x`.

In [4]:
let rec diff (e: Expr) : Expr =
    match e with
    | Add (f, g) -> 
        let fs, gs = diff f, diff g
        Add (fs, gs)
    | Sub (f, g) ->
        let fs, gs = diff f, diff g
        Sub (fs, gs)
    | Mul (f, g) ->
        let fs, gs = diff f, diff g
        Add (Mul (fs, g), Mul (f, gs))
    | Div (f, g) ->
        let fs, gs = diff f, diff g
        Div (Sub (Mul (fs, g), Mul (f, gs)), Mul (g, g))
    | Pow (f, g) ->
        diff (App ("exp", Mul (g, App ("ln", f))))
    | App ("ln", f) ->
        let fs = diff f
        Div (fs, f)
    | App ("exp", f) ->
        let fs = diff f
        Mul (fs, e)
    | Var "x" ->
        Num 1
    | _ ->
        Num 0

Let us inspect the details of this function:
<ol>
<li>  The case `Add (f, g)` implements the rule: 
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) + g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) + \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The case `Sub (f, g)` implements the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) - g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) - \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The case `Mul (f, g)` deals with products.  The 
      <a href="https://en.wikipedia.org/wiki/Product_rule">product rule</a> is      
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) \cdot g(x)\bigr) = \left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right)\cdot g(x) + f(x) \cdot \left(\frac{\mathrm{d}\;}{\mathrm{d}x} g(x)\right)
      $$
      </li>
<li>  The case `Div (f, g)` deals with quotients.  The
      <a href="https://en.wikipedia.org/wiki/Quotient_rule">quotient rule</a> is
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x}\left(\frac{f(x)}{g(x)}\right) = 
         \frac{\displaystyle\left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right)\cdot g(x) - 
         f(x) \cdot \left(\frac{\mathrm{d}\;}{\mathrm{d}x} g(x)\right)}{g(x) \cdot g(x)}
      $$
      </li>
<li>  The case `Pow (f, g)` deals with powers.  In order to take the derivative of an
      expression of the form
      $$  f(x)^{g(x)} $$
      we first need to rewrite this expression using the following trick:
      $$ f(x)^{g(x)} = \exp\bigl(\ln\bigl(f(x)^{g(x)}\bigr)\bigr) = \exp\bigl(g(x) \cdot \ln(f(x))\bigr) $$
      Then, we can recursively call `diff` for this expression.  This works, because the function
      `diff` can deal with both the exponential function $x \mapsto \exp(x)$ and with the natural
      logarithm $x \mapsto \ln(x)$.
      </li>
<li>  The case `App ("ln", f)` deals with expressions of the form 
      $$\ln\bigl(f(x)\bigr)$$  
      In order to take the derivative of this expression, we first need to know the derivative of the natural
      logarithm.  This derivative is given as     
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \ln(x) = \frac{1}{x}$$
      Then, using the <a href="https://en.wikipedia.org/wiki/Chain_rule">chain rule</a> we have that
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \ln\bigl(f(x)\bigr) = \frac{\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)}{f(x)}$$
      </li>
<li>  The case `App ("exp", f)` deals with expressions of the form $\exp\bigl(f(x)\bigr)$.  
      In order to take the derivative of this expression, we first need to know the derivative of the 
      <a href="https://en.wikipedia.org/wiki/Exponential_function">exponential function</a>.  
      This derivative is given as 
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \exp(x) = \exp(x)$$    
      Then, using the <a href="https://en.wikipedia.org/wiki/Chain_rule">chain rule</a> we have that
      $$\frac{\mathrm{d}\;}{\mathrm{d}x} \exp\bigl(f(x)\bigr) = \left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right) \cdot \exp\bigl(f(x)\bigr)
      $$
      </li>
<li>  The case `Var "x"` deals with the case where `e` is the variable `x`.   As we have
      $$\frac{\mathrm{d}x}{\mathrm{d}x} = 1,$$
      the function `diff` returns `Num 1` in this case.
      </li>
<li>  Otherwise, the expression is assumed to be a constant and hence we return `Num 0`.
      </li>
</ol>

Let us implement a small function `test`.  It takes one argument:
- `s` is a string representing an arithmetic expression.

It parses this string as an arithmetic expression, takes the derivative of this expression with respect to the variable `x`, and prints the result.  The function `toString` is defined in the notebook `Expr-Parser.ipynb`.

In [5]:
let test (s: string) : unit =
    let t = parse s
    let d = diff t
    printfn "d/dx %s = %s" s (toString d)

In [6]:
test "x ** x"

d/dx 

x ** x

 = 

(1*ln(x) + x*1/x)*exp(x*ln(x))

In [7]:
test "x * exp(x)"

d/dx 

x * exp(x)

 = 

1*exp(x) + x*1*exp(x)

In [8]:
test "ln(exp(x))"

d/dx 

ln(exp(x))

 = 

1*exp(x)/exp(x)

As we can see, the function works as advertised.  However, in order for this approach to be really useful we would have to implement a function `simplify` that takes an arithmetic expression `e` and then simplifies this expression using the laws of analysis.  Unfortunately, the implementation of that function is much harder than the implementation of the function `diff`.

In [9]:
test "ln(x)"

d/dx 

ln(x)

 = 

1/x